# 02_video.ipynb — UC-2 동영상 다운로드

`video()` 함수와 컨테이너 규칙, 재사용, 예외 검증.


In [ ]:
import os
import youtube_kit as yk

from pathlib import Path
# 루트의 outputs 에 저장한다. notebooks 에서 열어도, 루트에서 실행해도 같은 곳이다
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
out_dir = str(ROOT / "outputs")
os.makedirs(out_dir, exist_ok=True)


## 포맷 확인 및 다운로드 (T-13 Shorts)


In [2]:
# info() 결과에서 포맷 ID를 선택 (하드코딩하지 않음)
info_t13 = yk.info("JyEsEcsCqzM", out_dir=out_dir)
v_id = next(f["id"] for f in info_t13["video_formats"] if f["vcodec"].startswith("avc") or "h264" in f["vcodec"])
a_id = next(f["id"] for f in info_t13["audio_formats"] if f["ext"] == "m4a")
print(f"Selected formats from info: video={v_id}, audio={a_id}")

res_video = yk.video("JyEsEcsCqzM", video=v_id, audio=a_id, out_dir=out_dir)
print("video() result:", res_video)
assert res_video["status"] in ("ok", "reused")
assert res_video["path"] == os.path.join(out_dir, f"JyEsEcsCqzM.{v_id}_{a_id}.mp4")
assert os.path.exists(res_video["path"])


Selected formats from info: video=269, audio=139-drc
video() result: {'status': 'ok', 'video_id': 'JyEsEcsCqzM', 'url': 'https://www.youtube.com/watch?v=JyEsEcsCqzM', 'path': 'outputs\\JyEsEcsCqzM.269_139-drc.mp4'}


## 파일 재사용 확인


In [3]:
res_reused = yk.video("JyEsEcsCqzM", video=v_id, audio=a_id, out_dir=out_dir)
print("Reused result:", res_reused)
assert res_reused["status"] == "reused"
assert res_reused["path"] == res_video["path"]


Reused result: {'status': 'reused', 'video_id': 'JyEsEcsCqzM', 'url': 'https://www.youtube.com/watch?v=JyEsEcsCqzM', 'path': 'outputs\\JyEsEcsCqzM.269_139-drc.mp4'}


## 컨테이너 지정 (mkv) 및 overwrite 확인


In [4]:
res_mkv = yk.video("JyEsEcsCqzM", video=v_id, audio=a_id, container="mkv", out_dir=out_dir, overwrite=True)
print("mkv overwrite result:", res_mkv)
assert res_mkv["status"] == "ok"
assert res_mkv["path"] == os.path.join(out_dir, f"JyEsEcsCqzM.{v_id}_{a_id}.mkv")
assert os.path.exists(res_mkv["path"])


mkv overwrite result: {'status': 'ok', 'video_id': 'JyEsEcsCqzM', 'url': 'https://www.youtube.com/watch?v=JyEsEcsCqzM', 'path': 'outputs\\JyEsEcsCqzM.269_139-drc.mkv'}


## T-15: 라이브 영상 다운로드 시 NotAvailable


In [5]:
try:
    yk.video("rFZHOHl-L8A", video="137", audio="140", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("T-15 live stream rejected as expected:", type(exc).__name__)


T-15 live stream rejected as expected: NotAvailable


## 잘못된 컨테이너 이름 (InvalidInput)


In [6]:
try:
    yk.video("JyEsEcsCqzM", video=v_id, audio=a_id, container="avi", out_dir=out_dir)
    assert False, "Expected InvalidInput"
except yk.InvalidInput as exc:
    print("Invalid container rejected:", type(exc).__name__)


Invalid container rejected: InvalidInput


## 호환되지 않는 코덱을 mp4로 합치려 할 때 (NotAvailable)


In [7]:
info_t9 = yk.info("SbvAaDN1bpE", out_dir=out_dir)
vp9_id = next(f["id"] for f in info_t9["video_formats"] if "vp09" in f["vcodec"] or "vp9" in f["vcodec"])
opus_id = next(f["id"] for f in info_t9["audio_formats"] if "opus" in f["acodec"])
print(f"Selected incompatible codecs: video={vp9_id}, audio={opus_id}")

try:
    yk.video("SbvAaDN1bpE", video=vp9_id, audio=opus_id, container="mp4", out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("Incompatible codecs rejected for mp4:", type(exc).__name__)


Selected incompatible codecs: video=602, audio=249-drc
Incompatible codecs rejected for mp4: NotAvailable


## 없는 포맷 ID (NotAvailable)


In [8]:
try:
    yk.video("JyEsEcsCqzM", video="99999", audio=a_id, out_dir=out_dir)
    assert False, "Expected NotAvailable"
except yk.NotAvailable as exc:
    print("Non-existent format rejected:", type(exc).__name__)


Non-existent format rejected: NotAvailable


## 잘못된 입력 및 재생목록 URL (InvalidInput)


In [9]:
try:
    yk.video("https://www.youtube.com/playlist?list=PLyPc3E0Xm540Yfdln9qlVyyrqRME80jJm", video=v_id, audio=a_id, out_dir=out_dir)
    assert False, "Expected InvalidInput for playlist URL"
except yk.InvalidInput as exc:
    print("Playlist URL rejected in video():", type(exc).__name__)


Playlist URL rejected in video(): InvalidInput


## T-1 · T-10 컨테이너 규칙 (container 미지정)

H.264+AAC → mp4, VP9+Opus → mkv, AV1+AAC → mp4. 규칙은 코덱으로만 정해지므로 가장 낮은 해상도로 받는다.

In [ ]:
def lowest(formats, prefix):
    return min((f for f in formats if f["vcodec"].startswith(prefix)), key=lambda f: f["height"] or 0)

info_t1 = yk.info("bWPXADZylm0", out_dir=out_dir)
orig = [f for f in info_t1["audio_formats"] if f["original"] is True and not f["drc"]]
aac = next(f for f in orig if f["acodec"].startswith("mp4a"))
opus = next(f for f in orig if "opus" in f["acodec"])

avc = lowest(info_t1["video_formats"], "avc1")
r = yk.video("bWPXADZylm0", video=avc["id"], audio=aac["id"], out_dir=out_dir)
assert r["path"] == os.path.join(out_dir, f"bWPXADZylm0.{avc['id']}_{aac['id']}.mp4") and os.path.exists(r["path"])

vp9 = lowest(info_t1["video_formats"], "vp09")
r = yk.video("bWPXADZylm0", video=vp9["id"], audio=opus["id"], out_dir=out_dir)
assert r["path"] == os.path.join(out_dir, f"bWPXADZylm0.{vp9['id']}_{opus['id']}.mkv") and os.path.exists(r["path"])

info_t10 = yk.info("gTPUhQG4hqs", out_dir=out_dir)
av1 = lowest(info_t10["video_formats"], "av01")
aac10 = next(f for f in info_t10["audio_formats"] if f["acodec"].startswith("mp4a") and not f["drc"])
r = yk.video("gTPUhQG4hqs", video=av1["id"], audio=aac10["id"], out_dir=out_dir)
assert r["path"] == os.path.join(out_dir, f"gTPUhQG4hqs.{av1['id']}_{aac10['id']}.mp4") and os.path.exists(r["path"])
print("T-1 · T-10 container rule verified.")

## 임시 파일이 남지 않는다

In [ ]:
leftover = [n for n in os.listdir(out_dir) if n.startswith("tmp") or n.endswith((".part", ".ytdl", ".tmp"))]
assert not leftover, leftover
print("No temporary files left in", out_dir)